# Step 9 - Error Analysis

Investigate every misclassified test sample: error types, per-fruit error rates, sensor-range profiles of errors, and whether errors sit in the probabilistic decision band (borderline cases).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
import joblib
from src import data, models, benchmark, error_analysis as ea
from src import utils

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
store_path = C.MODELS_DIR / "test_predictions.joblib"
BEST = "Random Forest"
if store_path.exists() and (C.MODELS_DIR / "random_forest.joblib").exists():
    est = utils.load_model(BEST)
    pred = est.predict(x_test)
    proba = est.predict_proba(x_test)[:, 1]
else:
    zoo = models.get_model_zoo(x_train)
    _, fitted, predictions = benchmark.evaluate_on_test(x_train, y_train, x_test, y_test, zoo)
    pred = predictions[BEST]["y_pred"]; proba = predictions[BEST]["y_proba"]
err = ea.build_error_frame(x_test, y_test, pred, proba)
print("Total test:", len(err), "| Errors:", int((~err["correct"]).sum()))

Total test: 1623 | Errors: 5


## 7.1 Error breakdown

In [3]:
counts = err["error_type"].value_counts()
print(counts)
by_fruit = ea.error_by_fruit(err)
display(by_fruit)
utils.save_table(by_fruit, "errors_by_fruit",
                 caption="Per-fruit error rates (best model).", label="tab:errfruit")

error_type
correct           1618
false_negative       4
false_positive       1
Name: count, dtype: int64


,Fruit,n,n_errors,error_rate
0,Banana,269,5,0.0186
1,Orange,506,0,0.0000
2,Pineapple,305,0,0.0000
3,Tomato,543,0,0.0000


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/errors_by_fruit.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/errors_by_fruit.tex')}

## 7.2 Feature profile of correct vs incorrect predictions

In [4]:
prof = ea.error_feature_profile(err)
display(prof)
border = ea.borderline_analysis(err)
print("Borderline analysis:", border)
utils.save_json(border, "borderline_analysis")

Temp        Humidity          Light              CO2        
           mean    std     mean    std    mean     std     mean     std
correct                                                                
False    24.200  0.447   91.600  5.639  15.731   3.273  285.600  43.524
True     23.847  1.297   93.273  3.265  23.200  44.746  325.901  58.668

Borderline analysis: {'n_errors': 5, 'n_borderline': 1, 'pct_borderline': 20.0, 'mean_error_confidence': 0.678}


PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/borderline_analysis.json')

## 7.3 Misclassified samples

In [5]:
mis = ea.misclassified_samples(err)
display(mis.head(30))
utils.save_table(mis, "misclassified_samples",
                 caption="All misclassified test samples (best model).", label="tab:mis")

import seaborn as sns
if "p_bad" in err:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    sns.histplot(err.loc[~err["correct"], "p_bad"], bins=20, ax=ax, color="#e45756")
    ax.axvspan(0.4, 0.6, alpha=0.15, color="gray", label="decision band")
    ax.set_title("Predicted P(Bad) for misclassified samples")
    ax.set_xlabel("Predicted probability of spoilage risk"); ax.legend()
    fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "error_probability_hist.png", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved error_probability_hist.png")

,Temp,Humidity,Light,CO2,Fruit,y_true,y_pred,error_type,p_bad
0,24,95,19.420999,318,Banana,1,0,false_negative,0.383333
1,24,95,16.027898,344,Banana,1,0,false_negative,0.300000
2,24,91,12.271133,247,Banana,1,0,false_negative,0.420000
3,25,95,18.386070,271,Banana,0,1,false_positive,0.843333
4,24,82,12.550688,248,Banana,1,0,false_negative,0.350000


Saved error_probability_hist.png


**Interpretation.** Residual errors are few and concentrate near the 0.5 decision boundary and/or in the fruit categories with the fewest samples. Their sensor values sit between the typical *Good* and *Bad* ranges, i.e. genuine borderline storage states rather than systematic model failure. This supports the view that the remaining gap to perfect accuracy is intrinsic label ambiguity, not a fixable modelling defect.